# Experiment Mobile Controller

Load a mobile camera controller configuration and inspect which observation dome latitude/longitude positions are reachable.

In [1]:
import sys
sys.path.append("..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

from interbotix_common_modules.common_robot.robot import robot_shutdown, robot_startup
from interbotix_xs_modules.xs_robot.arm import InterbotixManipulatorXS
from mobile_camera.observation_dome import ObservationDome
from mobile_camera.widowx_observation_dome import WidowX_ObservationDomeDriver


experiment = "controllers"
run = "mobile_camera_controller_00"
exp = Config().get_experiment(experiment, run)

dome_config = exp["observation_dome"]
dome = ObservationDome(
    x=dome_config["x"],
    y=dome_config["y"],
    z=dome_config["z"],
    r=dome_config["r"],
    no_long=dome_config["no_long"],
    no_lat=dome_config["no_lat"],
)

bot = InterbotixManipulatorXS(
    robot_model=exp.get("robot_model", "wx250s"),
    group_name=exp.get("group_name", "arm"),
    gripper_name=exp.get("gripper_name", "gripper"),
)
robot_startup()

dome_driver = WidowX_ObservationDomeDriver(
    bot.arm,
    dome,
    safety_box=exp.get("safety_box"),
)
reachability_kwargs = exp.get("decision", {}).get("reachability_kwargs", {})


def reachable_report(lat_index, long_index):
    return dome_driver.is_lat_long_reachable(
        lat_index,
        long_index,
        return_report=True,
        **reachability_kwargs,
    )


try:
    reachable_positions = []
    print(f"Loaded exp/run {experiment}/{run}: {exp.get('controller_name', '<unnamed controller>')}")
    print("Reachable lat/long positions:")

    for lat_index in range(dome.no_lat + 1):
        long_indices = [0] if lat_index == 0 else range(dome.no_long)
        for long_index in long_indices:
            report = reachable_report(lat_index, long_index)
            if report["safe"] and report["reachable"]:
                reachable_positions.append((lat_index, long_index))
                x, y, z = dome.get_lat_long_point(lat_index, long_index)
                print(f"  lat={lat_index:2d}, long={long_index:2d} -> x={x:.4f}, y={y:.4f}, z={z:.4f}")

    if not reachable_positions:
        print("  <none>")

    default_position = (
        exp.get("default_lat_index", 0),
        exp.get("default_long_index", 0),
    )
    decision_config = exp.get("decision", {})
    middle_position = (
        decision_config.get("middle_lat_index", dome.no_lat // 2),
        decision_config.get("middle_long_index", 0),
    )

    for label, position in (
        ("default", default_position),
        ("middle", middle_position),
    ):
        lat_index, long_index = position
        report = reachable_report(lat_index, long_index)
        reachable = report["safe"] and report["reachable"]
        print(f"{label.title()} position lat={lat_index}, long={long_index} reachable: {reachable} ({report['status']})")
finally:
    robot_shutdown(bot.core.get_node())


***ExpRun**: Loading pointer config file:
	/home/lboloni/.config/BerryPicker/mainsettings.yaml
***ExpRun**: Loading machine-specific config file:
	/home/lboloni/Insync/lotzi.boloni@gmail.com/Google Drive/LotziStudy/Code/PackageTracking/BerryPicker/settings/settings-tredy2.yaml
***ExpRun**: Using torch device: cuda
***ExpRun**: Experiment default config /home/lboloni/Documents/Hackingwork/_Checkouts/BerryPicker/BerryPicker/src/experiment_configs/controllers/_defaults_controllers.yaml was empty, ok.
***ExpRun**: Configuration for exp/run: controllers/mobile_camera_controller_00 successfully loaded


[INFO] [1777918675.640902534] [interbotix_robot_manipulation]: Initialized InterbotixRobotNode!
[INFO] [1777918677.756468912] [interbotix_robot_manipulation]: 
	Robot Name: wx250s
	Robot Model: wx250s
[INFO] [1777918677.757605252] [interbotix_robot_manipulation]: Initialized InterbotixRobotXSCore!
[INFO] [1777918677.762195566] [interbotix_robot_manipulation]: 
	Arm Group Name: arm
	Moving Time: 2.00 seconds
	Acceleration Time: 0.30 seconds
	Drive Mode: Time-Based-Profile
[INFO] [1777918677.763126054] [interbotix_robot_manipulation]: Initialized InterbotixArmXSInterface!
[INFO] [1777918678.265974621] [interbotix_robot_manipulation]: 
	Gripper Name: gripper
	Gripper Pressure: 50.0%
[INFO] [1777918678.267269402] [interbotix_robot_manipulation]: Initialized InterbotixGripperXSInterface!
[WARN] [1777918678.410245044] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute


Loaded exp/run controllers/mobile_camera_controller_00: mobile_camera_controller
Reachable lat/long positions:
  lat= 0, long= 0 -> x=0.2500, y=0.0000, z=0.2500
  lat= 1, long= 1 -> x=0.2563, y=0.0046, z=0.2494
  lat= 1, long= 2 -> x=0.2524, y=0.0074, z=0.2494
  lat= 1, long= 3 -> x=0.2476, y=0.0074, z=0.2494
  lat= 1, long= 4 -> x=0.2437, y=0.0046, z=0.2494
  lat= 1, long= 5 -> x=0.2422, y=0.0000, z=0.2494
  lat= 1, long= 6 -> x=0.2437, y=-0.0046, z=0.2494
  lat= 1, long= 7 -> x=0.2476, y=-0.0074, z=0.2494
  lat= 1, long= 8 -> x=0.2524, y=-0.0074, z=0.2494
  lat= 1, long= 9 -> x=0.2563, y=-0.0046, z=0.2494


[WARN] [1777918678.860221186] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute
[WARN] [1777918678.963293049] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute
[WARN] [1777918679.046620787] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute


  lat= 2, long= 3 -> x=0.2452, y=0.0147, z=0.2476
  lat= 2, long= 4 -> x=0.2375, y=0.0091, z=0.2476
  lat= 2, long= 5 -> x=0.2345, y=0.0000, z=0.2476
  lat= 2, long= 6 -> x=0.2375, y=-0.0091, z=0.2476
  lat= 2, long= 7 -> x=0.2452, y=-0.0147, z=0.2476


...

[WARN] [1777918684.993816512] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute
[WARN] [1777918685.126008691] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute


  lat=10, long= 7 -> x=0.2345, y=-0.0476, z=0.2000
Default position lat=0, long=0 reachable: True (reached)


[WARN] [1777918685.291046626] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute
[WARN] [1777918685.400443852] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute


Middle position lat=5, long=0 reachable: False (unreachable)


[WARN] [1777918685.533701066] [interbotix_robot_manipulation]: No valid pose could be found. Will not execute
